# Non-2023 Hybrid Experiment

## Protocol

**Status:** fixed on 2026-10-16, before any model in this notebook was run. The protocol was written after notebooks 06–08 and is recorded in git before execution.

### 1. Framing

This is a **pre-specified sensitivity analysis restricted to non-2023 tissues**. It is **not** an independent confirmation.

The hypothesis comes from an exploratory result in `08_missing_information.ipynb` (Section 10.2): on non-2023 tissues, hybrid minus tabular macro F1 was +0.036 [−0.004, +0.075], p = 0.078. That result used the same tissues, folds, embeddings and classifier. This analysis differs in two respects only:

1. Models are **trained** on non-2023 tissues only (in notebook 08 they were trained on all years).
2. Year is defined from `Tissue_ID` (notebook 08 used the Konan exam year first, which placed 3 tissues with a 2023 ID in other years).

The question is whether the exploratory signal persists when training excludes the 2023 subset. A similar result will be reported as consistent with notebook 08, not as confirmation.

### 2. Research question

Do specular microscopy image features provide additional information for expert donor cornea grading beyond the available tabular clinical and morphological features, when evaluation is restricted to tissues from years other than 2023?

### 3. Cohort

| Criterion | Definition |
|---|---|
| Labelled | `Tissue_Grade_Clean` in {A+, A, B} (`ms.load_labelled_data()`) |
| Usable image | Present in `clean_image_manifest.csv`, excluding the two tissues flagged for manual review (`NEB-2024-08-16259 R`, `NEB-2025-09-17471 R`) |
| Year | Four-digit year in `Tissue_ID` (`NEB-YYYY-…`) |
| Eligible | Year ≠ 2023 |

**Expected cohort:** 551 tissues from 312 donors; A+ 277, A 196, B 78. By year: 2020: 128, 2021: 12, 2022: 58, 2024: 168, 2025: 185. The notebook asserts these counts and stops if any differ.

### 4. Features and models

| Element | Setting |
|---|---|
| Tabular features | The 24 features in `ms.ALL_FEATURES`, preprocessed with `ms.make_preprocessor()` |
| Image features | Cached `resnet50_imagenet_fullres.npz` (frozen ImageNet ResNet-50, full resolution, 2048-d); no re-extraction |
| Image reduction | StandardScaler → PCA, 16 components, `random_state=42` |
| Model A: Tabular | Tabular preprocessing → Random Forest |
| Model B: Hybrid | Tabular preprocessing + image reduction → Random Forest |
| Random Forest | `n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=1, max_features="sqrt", class_weight="balanced", random_state=42` (identical for both models; no tuning) |

### 5. Validation

- Existing folds `Fold_r0` … `Fold_r4` from `donor_grouped_folds.csv`, restricted to the eligible cohort. Training and test tissues in every fold are non-2023 only.
- All preprocessing and PCA are fitted on the training fold only, through a cloned pipeline.
- Checks, which must pass before any model is fitted:
  - no donor appears in both the training and test sets of any fold;
  - every eligible tissue is tested exactly once per repeat;
  - every test fold contains at least one tissue of each grade.

### 6. Metrics

| Role | Metric | Level |
|---|---|---|
| **Primary** | Macro F1 | Per fold (25 paired values) |
| Secondary | Balanced accuracy, accuracy, grade B recall | Per fold |
| Descriptive | Per-grade precision and recall | Pooled out-of-fold predictions over all repeats |
| Descriptive | Repeat-level macro F1 | Pooled out-of-fold predictions per repeat |

### 7. Statistical analysis

- **Primary comparison:** hybrid minus tabular macro F1, paired by fold.
- Report the mean paired difference, a corrected 95% confidence interval and a corrected paired t-test (Nadeau & Bengio, 2003; Bouckaert & Frank, 2004). The variance is inflated by n_test / n_train using mean fold sizes, with 24 degrees of freedom.
- Secondary metrics use the same test. They are reported unadjusted and labelled secondary; no claim rests on them.
- Repeat-level differences are descriptive. Repeats reuse the same tissues and are not treated as independent.

**Per-year results (descriptive):**

- Using pooled out-of-fold predictions over all repeats, report tabular macro F1, hybrid macro F1 and their difference for each year (2020, 2021, 2022, 2024, 2025), with tissue counts.
- Report the overall difference recomputed with each year excluded in turn, to show whether any single year drives the result.
- 2021 (12 tissues) is shown but flagged as too small to interpret. No significance tests are run per year.

### 8. Interpretation rules

Fixed before running; the conclusion is read from the primary comparison only.

| Primary result (macro F1 difference) | Reported as |
|---|---|
| 95% CI lower bound > 0 | Hybrid outperformed tabular in this sensitivity analysis |
| 95% CI includes 0 | Inconclusive: no evidence of a difference |
| 95% CI upper bound < 0 | Hybrid underperformed tabular |

**Additional rules:**

- **Grade B recall qualifier:** if the mean fold-level grade B recall difference is below −0.02, the result is reported together with the reduced B recall, and the hybrid is not recommended for use, whatever the macro F1 result. The −0.02 threshold is a project-defined decision rule fixed for this analysis, not a clinically validated safety threshold.
- **Per-year qualifier:** if excluding any single year turns a positive overall difference to zero or below, the result is reported as dependent on that year.
- **Framing:** in every case the result is described as a sensitivity analysis, not independent confirmation of notebook 08.
- **No changes after results:** no alternative PCA sizes, image models, classifiers, hyperparameters, cohorts or metrics. Any further analysis is labelled post hoc.
- **Errors:** a code error may be fixed and the notebook rerun only if the fix does not change a protocol choice. Every rerun is recorded in the deviations log with its reason.

### 9. Proposed future validation plan

This section is a proposed plan, not evidence that future data will be available or independent. It applies only if a genuinely new cohort is obtained.

If such a cohort becomes available, the same protocol will be applied unchanged to it. The cohort must consist of independently collected and graded tissues: those not present in `cleaned_grading_dataset.xlsx` at commit `6e9c460`, excluding any from 2023.

- Both models are fitted once on all eligible tissues from the current cohort and evaluated once on the new cohort.
- Uncertainty for the paired macro F1 difference comes from a donor-level bootstrap (2,000 resamples, seed 42).
- The interpretation rules in Section 8 apply.
- Only that test, on a cohort verified as new and independently graded, would count as confirmatory.

## Deviations Log

Record any rerun or departure from the protocol here, with date and reason, before reporting results.

| Date | Change | Reason |
|---|---|---|
| — | None | — |